# 03 — LT, Poisson and XGBoost model selection

Validation-stage model development only. The held-out 2023–2025 test set is
not used for selection or tuning.

The final tuned 4-window versus 5-window XGBoost bootstrap is isolated in
notebook `04_xgboost_4_vs_5_tuned_bootstrap_check.ipynb`.

In [ ]:
from pathlib import Path
from itertools import combinations
import gc, itertools, time
import numpy as np
import pandas as pd
from scipy.special import gammaln
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import MaxAbsScaler
import xgboost as xgb

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

GRID_MASK_FILE = DATA_DIR / "california_grid_centre_mask.csv"
FINAL_CATALOGUE_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_Mw25_centre_mask.csv"

print("XGBoost version:", xgb.__version__)
if xgb.__version__ != "3.4.1":
    raise RuntimeError(
        "Reported XGBoost selection results require xgboost==3.4.1; "
        f"found {xgb.__version__}."
    )

## 1. Common target, predictors and temporal split

In [ ]:
grid_mask = pd.read_csv(GRID_MASK_FILE)
df = pd.read_csv(FINAL_CATALOGUE_FILE)

df["time"] = pd.to_datetime(df["time"], format="mixed", utc=True)
df["date"] = df["time"].dt.floor("D").dt.tz_localize(None)

dates = pd.date_range("2010-01-01", "2025-12-31", freq="D")
cell_ids = np.sort(grid_mask["cell_id"].astype(int).unique())

assert len(df) == 30347
assert len(cell_ids) == 1080

daily_counts_25 = (
    df.groupby(["date", "cell_id"]).size()
      .unstack(fill_value=0)
      .reindex(index=dates, columns=cell_ids, fill_value=0)
)

target_df = df.loc[df["mag_Mw"] >= 3.0].copy()
daily_counts_target = (
    target_df.groupby(["date", "cell_id"]).size()
             .unstack(fill_value=0)
             .reindex(index=dates, columns=cell_ids, fill_value=0)
)

Y_7 = sum(daily_counts_target.shift(-h) for h in range(1, 8))

candidate_windows = [1, 3, 7, 10, 14, 21, 30]
X_windows = {
    w: daily_counts_25.rolling(window=w, min_periods=w).sum()
    for w in candidate_windows
}
X_1, X_7 = X_windows[1], X_windows[7]

eligible = X_windows[30].notna().all(axis=1) & Y_7.notna().all(axis=1)
forecast_dates = dates[eligible]

train_dates = forecast_dates[
    (forecast_dates >= "2010-01-30") & (forecast_dates <= "2019-12-24")
]
val_dates = forecast_dates[
    (forecast_dates >= "2020-01-01") & (forecast_dates <= "2022-12-24")
]
test_dates = forecast_dates[
    (forecast_dates >= "2023-01-01") & (forecast_dates <= "2025-12-24")
]

assert len(train_dates) == 3616
assert len(val_dates) == 1089
assert len(test_dates) == 1089

Y_val, Y_test = Y_7.loc[val_dates], Y_7.loc[test_dates]

def flatten_target(origin_dates):
    return Y_7.loc[origin_dates].to_numpy(dtype=float).reshape(-1)

y_train = flatten_target(train_dates)
y_val = flatten_target(val_dates)

def poisson_log_score_values(y, lam):
    y = np.asarray(y, dtype=np.float64)
    lam = np.clip(np.asarray(lam, dtype=np.float64), 1e-15, None)
    return y * np.log(lam) - lam - gammaln(y + 1.0)

def poisson_log_score(y, lam):
    return poisson_log_score_values(y, lam).mean()

def moving_block_bootstrap_mean(values, block_length, n_boot=5000, seed=2026):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=np.float64)
    n = len(values)
    starts = np.arange(0, n - block_length + 1)
    n_blocks = int(np.ceil(n / block_length))
    boot = np.empty(n_boot)
    for b in range(n_boot):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        sample = np.concatenate([values[s:s+block_length] for s in chosen])[:n]
        boot[b] = sample.mean()
    return boot

print("Target Mw*>=3.0 events:", int(daily_counts_target.to_numpy().sum()))
print("Origins:", len(train_dates), len(val_dates), len(test_dates))

## 2. Long-term spatial rate and smoothing selection

In [ ]:
# =========================================================
# 21. Fit Long-Term Spatial Rate model
# =========================================================

TRAIN_START = pd.Timestamp("2010-01-01")
TRAIN_END = pd.Timestamp("2019-12-31")

# Daily Mw* >= 3.0 target-earthquake counts during training period
train_daily_target = daily_counts_target.loc[
    TRAIN_START:TRAIN_END
].copy()

n_train_days = len(train_daily_target)

# Mean daily target-earthquake rate in each cell
r_hat = (
    train_daily_target.sum(axis=0)
    / n_train_days
)

# Convert daily expected rate to 7-day expected count
lambda_LT = 7 * r_hat

print("Number of training calendar days:", n_train_days)
print(
    "Total Mw* >= 3.0 earthquakes in training period:",
    int(train_daily_target.to_numpy().sum())
)
print(
    "Cells with non-zero long-term rate:",
    int((r_hat > 0).sum())
)
print(
    "Cells with zero long-term rate:",
    int((r_hat == 0).sum())
)

print("\nMean daily rate across cells:", r_hat.mean())
print("Maximum daily rate:", r_hat.max())

print("\nMean 7-day LT forecast across cells:", lambda_LT.mean())
print("Maximum 7-day LT forecast:", lambda_LT.max())
print(
    "Total expected earthquakes across California per 7 days:",
    lambda_LT.sum()
)

In [ ]:
# =========================================================
# 22. Sanity checks for LT model
# =========================================================

training_total = train_daily_target.to_numpy().sum()

# Sum of cell-specific daily rates should equal
# overall mean number of target earthquakes per training day
expected_daily_total = training_total / n_train_days

print(
    "Observed mean earthquakes per training day:",
    expected_daily_total
)

print(
    "Sum of estimated cell daily rates:",
    r_hat.sum()
)

assert np.isclose(
    r_hat.sum(),
    expected_daily_total
)

assert np.isclose(
    lambda_LT.sum(),
    7 * expected_daily_total
)

print("\nAll LT rate checks passed.")

In [ ]:
# =========================================================
# 23. Generate LT forecasts for validation and test periods
# =========================================================

LT_val = pd.DataFrame(
    np.tile(
        lambda_LT.to_numpy(),
        (len(val_dates), 1)
    ),
    index=val_dates,
    columns=cell_ids
)

LT_test = pd.DataFrame(
    np.tile(
        lambda_LT.to_numpy(),
        (len(test_dates), 1)
    ),
    index=test_dates,
    columns=cell_ids
)

# Corresponding observed 7-day targets
Y_val = Y_7.loc[val_dates]
Y_test = Y_7.loc[test_dates]

print("LT validation forecast:", LT_val.shape)
print("Validation target:", Y_val.shape)

print("\nLT test forecast:", LT_test.shape)
print("Test target:", Y_test.shape)

assert LT_val.shape == Y_val.shape
assert LT_test.shape == Y_test.shape

print("\nForecast and target dimensions match.")

In [ ]:
# =========================================================
# 24. Check zero-rate cells
# =========================================================

zero_rate_cells = lambda_LT[lambda_LT == 0].index

print("Number of zero-rate cells:", len(zero_rate_cells))

# Did any validation/test earthquakes occur in cells
# whose training LT rate was zero?
val_events_in_zero_cells = (
    Y_val[zero_rate_cells].to_numpy().sum()
)

test_events_in_zero_cells = (
    Y_test[zero_rate_cells].to_numpy().sum()
)

print(
    "Observed validation target counts in zero-rate cells:",
    val_events_in_zero_cells
)

print(
    "Observed test target counts in zero-rate cells:",
    test_events_in_zero_cells
)

In [ ]:
# =========================================================
# 25. Diagnose future activity in zero-rate training cells
# =========================================================

# The first validation/test forecast origin is Jan 1,
# therefore the first target day is Jan 2.
val_eval_daily = daily_counts_target.loc[
    "2020-01-02":"2022-12-31",
    zero_rate_cells
]

test_eval_daily = daily_counts_target.loc[
    "2023-01-02":"2025-12-31",
    zero_rate_cells
]

val_unique_events_zero = int(
    val_eval_daily.to_numpy().sum()
)

test_unique_events_zero = int(
    test_eval_daily.to_numpy().sum()
)

val_new_active_cells = int(
    (val_eval_daily.sum(axis=0) > 0).sum()
)

test_new_active_cells = int(
    (test_eval_daily.sum(axis=0) > 0).sum()
)

print(
    "Unique validation-period earthquakes in training-zero cells:",
    val_unique_events_zero
)

print(
    "Training-zero cells active during validation:",
    val_new_active_cells
)

print(
    "Unique test-period earthquakes in training-zero cells:",
    test_unique_events_zero
)

print(
    "Training-zero cells active during test:",
    test_new_active_cells
)

In [ ]:
# =========================================================
# 26. Tune LT spatial smoothing on validation data
# =========================================================

from scipy.special import gammaln

# Training counts by cell
N_g = train_daily_target.sum(axis=0).astype(float)

N_total = N_g.sum()
G = len(cell_ids)

# Empirical spatial distribution
p_empirical = N_g / N_total

# Overall expected number of target earthquakes per 7 days
weekly_total_rate = 7 * N_total / n_train_days

print("Training target earthquakes:", N_total)
print("Number of cells:", G)
print("Expected California-wide 7-day count:", weekly_total_rate)


def smoothed_LT_rate(epsilon):
    """
    Long-term spatial rate with shrinkage towards
    a uniform spatial background.
    """
    p_smoothed = (
        (1 - epsilon) * p_empirical
        + epsilon / G
    )

    return weekly_total_rate * p_smoothed


def mean_poisson_log_score(Y, lambda_vec):
    """
    Mean Poisson log score over all day-cell observations.
    Higher is better.
    """

    y = Y.to_numpy(dtype=float)

    lam = np.broadcast_to(
        lambda_vec.to_numpy()[None, :],
        y.shape
    )

    score = (
        y * np.log(lam)
        - lam
        - gammaln(y + 1)
    )

    return score.mean()

In [ ]:
epsilon_grid = np.concatenate([
    np.logspace(-6, -1, 26),
    np.array([0.2, 0.5, 1.0])
])

smoothing_results = []

for eps in epsilon_grid:

    lam = smoothed_LT_rate(eps)

    score = mean_poisson_log_score(
        Y_val,
        lam
    )

    smoothing_results.append({
        "epsilon": eps,
        "mean_validation_log_score": score,
        "min_cell_rate": lam.min(),
        "max_cell_rate": lam.max(),
        "total_7day_rate": lam.sum()
    })

smoothing_results = pd.DataFrame(
    smoothing_results
).sort_values(
    "mean_validation_log_score",
    ascending=False
)

smoothing_results.head(10)

In [ ]:
best_epsilon = smoothing_results.iloc[0]["epsilon"]

lambda_LT_smoothed = smoothed_LT_rate(
    best_epsilon
)

print("Best epsilon:", best_epsilon)
print(
    "Best validation mean log score:",
    smoothing_results.iloc[0]["mean_validation_log_score"]
)

print(
    "Minimum smoothed cell rate:",
    lambda_LT_smoothed.min()
)

print(
    "Maximum smoothed cell rate:",
    lambda_LT_smoothed.max()
)

print(
    "Total 7-day rate:",
    lambda_LT_smoothed.sum()
)

assert (lambda_LT_smoothed > 0).all()

assert np.isclose(
    lambda_LT_smoothed.sum(),
    weekly_total_rate
)

print("\nSmoothed LT checks passed.")

In [ ]:
# =========================================================
# 28. Finalise the Long-Term Spatial Rate benchmark
# =========================================================

EPSILON_LT = 0.10

lambda_LT_final = smoothed_LT_rate(EPSILON_LT)

print("Final LT smoothing epsilon:", EPSILON_LT)
print("Minimum 7-day cell forecast:", lambda_LT_final.min())
print("Maximum 7-day cell forecast:", lambda_LT_final.max())
print("California-wide expected count per 7 days:",
      lambda_LT_final.sum())

assert (lambda_LT_final > 0).all()
assert np.isclose(
    lambda_LT_final.sum(),
    weekly_total_rate
)

print("\nFinal LT benchmark locked.")

In [ ]:
lambda_LT_reference = lambda_LT_final.copy()

def repeat_LT(origin_dates):
    return np.tile(lambda_LT_reference.to_numpy(dtype=float), len(origin_dates))

LT_train = repeat_LT(train_dates)
LT_val = repeat_LT(val_dates)

print("Final epsilon:", EPSILON_LT)
print("LT statewide 7-day total:", lambda_LT_reference.sum())

## 3. Poisson recent-window selection

In [ ]:
def get_log_window(w, origin_dates):
    return np.log1p(
        X_windows[w].loc[origin_dates].to_numpy(dtype=float).reshape(-1)
    )

candidate_sets = []
for r in [1, 2, 3]:
    candidate_sets.extend(combinations(candidate_windows, r))

rows, models = [], {}
for windows in candidate_sets:
    Xtr = np.column_stack([get_log_window(w, train_dates) for w in windows])
    Xva = np.column_stack([get_log_window(w, val_dates) for w in windows])

    model = PoissonRegressor(
        alpha=0.0, fit_intercept=True, max_iter=3000, tol=1e-9
    )
    model.fit(Xtr, y_train / LT_train, sample_weight=LT_train)
    lam = LT_val * model.predict(Xva)

    name = "+".join(map(str, windows))
    rows.append({
        "windows": name,
        "n_windows": len(windows),
        "validation_log_score": poisson_log_score(y_val, lam),
        "iterations": model.n_iter_
    })
    models[name] = {"model": model, "lambda_val": lam}

poisson_window_comparison = (
    pd.DataFrame(rows).sort_values("validation_log_score", ascending=False)
)
display(poisson_window_comparison.head(15))
poisson_window_comparison.to_csv(
    TABLE_DIR / "poisson_window_selection.csv", index=False
)

lam_137 = models["1+3+7"]["lambda_val"]
lam_17 = models["1+7"]["lambda_val"]
point_diff = (
    poisson_log_score_values(y_val, lam_137)
    - poisson_log_score_values(y_val, lam_17)
)
origin_diff = point_diff.reshape(len(val_dates), len(cell_ids)).mean(axis=1)

boot_rows = []
for b in [7,14,30]:
    boot = moving_block_bootstrap_mean(origin_diff, b)
    lo, hi = np.quantile(boot, [0.025,0.975])
    boot_rows.append({
        "block_length": b,
        "mean_difference_137_minus_17": origin_diff.mean(),
        "ci_lower": lo,
        "ci_upper": hi,
        "ci_includes_zero": bool(lo <= 0 <= hi)
    })
poisson_window_bootstrap = pd.DataFrame(boot_rows)
display(poisson_window_bootstrap)
poisson_window_bootstrap.to_csv(
    TABLE_DIR / "poisson_137_vs_17_bootstrap.csv", index=False
)

## 4. Poisson long-term treatment and raw/log recent counts

In [ ]:
# =========================================================
# Final consistency check:
# 1-day + 7-day predictors only
# =========================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import MaxAbsScaler


# ---------------------------------------------------------
# Prepare 1-day and 7-day predictors
# ---------------------------------------------------------

# Raw
X_train_17_raw = np.column_stack([
    X_1.loc[train_dates].to_numpy(dtype=float).reshape(-1),
    X_7.loc[train_dates].to_numpy(dtype=float).reshape(-1)
])

X_val_17_raw = np.column_stack([
    X_1.loc[val_dates].to_numpy(dtype=float).reshape(-1),
    X_7.loc[val_dates].to_numpy(dtype=float).reshape(-1)
])

# Log-transformed
X_train_17_log = np.log1p(X_train_17_raw)
X_val_17_log = np.log1p(X_val_17_raw)

log_LT_train = np.log(LT_train)
log_LT_val = np.log(LT_val)


# ---------------------------------------------------------
# Function for LOG models
# ---------------------------------------------------------

def fit_17_log(spec):

    if spec == "no_LT":
        Xtr = X_train_17_log
        Xva = X_val_17_log

        model = PoissonRegressor(
            alpha=0.0,
            fit_intercept=True,
            max_iter=2000,
            tol=1e-9
        )
        model.fit(Xtr, y_train)
        lam = model.predict(Xva)

    elif spec == "LT_predictor":
        Xtr = np.column_stack([
            log_LT_train,
            X_train_17_log
        ])
        Xva = np.column_stack([
            log_LT_val,
            X_val_17_log
        ])

        model = PoissonRegressor(
            alpha=0.0,
            fit_intercept=True,
            max_iter=2000,
            tol=1e-9
        )
        model.fit(Xtr, y_train)
        lam = model.predict(Xva)

    elif spec == "LT_offset":
        model = PoissonRegressor(
            alpha=0.0,
            fit_intercept=True,
            max_iter=2000,
            tol=1e-9
        )

        model.fit(
            X_train_17_log,
            y_train / LT_train,
            sample_weight=LT_train
        )

        lam = LT_val * model.predict(
            X_val_17_log
        )

    return model, lam


# ---------------------------------------------------------
# Function for RAW models
# ---------------------------------------------------------

scaler_17 = MaxAbsScaler()

X_train_17_raw_scaled = scaler_17.fit_transform(
    X_train_17_raw
)

X_val_17_raw_scaled = scaler_17.transform(
    X_val_17_raw
)


def fit_17_raw(spec):

    if spec == "no_LT":
        Xtr = X_train_17_raw_scaled
        Xva = X_val_17_raw_scaled

        model = PoissonRegressor(
            alpha=0.0,
            fit_intercept=True,
            max_iter=3000,
            tol=1e-9
        )
        model.fit(Xtr, y_train)
        lam = model.predict(Xva)

    elif spec == "LT_predictor":
        Xtr = np.column_stack([
            log_LT_train,
            X_train_17_raw_scaled
        ])
        Xva = np.column_stack([
            log_LT_val,
            X_val_17_raw_scaled
        ])

        model = PoissonRegressor(
            alpha=0.0,
            fit_intercept=True,
            max_iter=3000,
            tol=1e-9
        )
        model.fit(Xtr, y_train)
        lam = model.predict(Xva)

    elif spec == "LT_offset":
        model = PoissonRegressor(
            alpha=0.0,
            fit_intercept=True,
            max_iter=3000,
            tol=1e-9
        )

        model.fit(
            X_train_17_raw_scaled,
            y_train / LT_train,
            sample_weight=LT_train
        )

        lam = LT_val * model.predict(
            X_val_17_raw_scaled
        )

    return model, lam


# ---------------------------------------------------------
# Fit all six
# ---------------------------------------------------------

rows = []
final_17_models = {}

for spec in ["no_LT", "LT_predictor", "LT_offset"]:

    # Log
    model, lam = fit_17_log(spec)

    rows.append({
        "model": f"{spec}_log",
        "LT_treatment": spec,
        "recent_form": "log1p",
        "validation_log_score":
            poisson_log_score(y_val, lam),
        "iterations": model.n_iter_
    })

    final_17_models[f"{spec}_log"] = {
        "model": model,
        "lambda_val": lam
    }

    # Raw
    model, lam = fit_17_raw(spec)

    rows.append({
        "model": f"{spec}_raw",
        "LT_treatment": spec,
        "recent_form": "raw",
        "validation_log_score":
            poisson_log_score(y_val, lam),
        "iterations": model.n_iter_
    })

    final_17_models[f"{spec}_raw"] = {
        "model": model,
        "lambda_val": lam
    }


final_17_comparison = (
    pd.DataFrame(rows)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

print(final_17_comparison.to_string(index=False))


# LT coefficient of the final predictor-log model
final_model = final_17_models[
    "LT_predictor_log"
]["model"]

print("\nLT predictor + log(1+X), 1+7 windows:")
print("Intercept:", final_model.intercept_)
print("beta_LT:", final_model.coef_[0])
print("beta_1:", final_model.coef_[1])
print("beta_7:", final_model.coef_[2])

In [ ]:
# =========================================================
# Final paired comparison:
# LT offset vs LT predictor using final 1+7 windows
# =========================================================

lambda_offset_17 = final_17_models[
    "LT_offset_log"
]["lambda_val"]

lambda_predictor_17 = final_17_models[
    "LT_predictor_log"
]["lambda_val"]


# Pointwise Poisson log scores
score_offset_17 = poisson_log_score_values(
    y_val,
    lambda_offset_17
)

score_predictor_17 = poisson_log_score_values(
    y_val,
    lambda_predictor_17
)


# Positive = offset model performs better
score_difference = (
    score_offset_17 - score_predictor_17
)

difference_matrix = score_difference.reshape(
    len(val_dates),
    len(cell_ids)
)

daily_difference = difference_matrix.mean(axis=1)


print("Overall mean difference:")
print(daily_difference.mean())

print("\nMedian daily difference:")
print(np.median(daily_difference))

print("\nProportion of forecast origins where offset is better:")
print(np.mean(daily_difference > 0))

print("\nDaily difference quantiles:")
print(
    pd.Series(daily_difference).quantile(
        [0, 0.05, 0.25, 0.5,
         0.75, 0.95, 1]
    )
)

In [ ]:
bootstrap_results_offset = []

for block_length in [7, 14, 30]:

    boot = moving_block_bootstrap_mean(
        daily_difference,
        block_length=block_length,
        n_boot=5000
    )

    lower, upper = np.quantile(
        boot,
        [0.025, 0.975]
    )

    bootstrap_results_offset.append({
        "block_length_days": block_length,
        "observed_mean_difference":
            daily_difference.mean(),
        "CI_2.5%": lower,
        "CI_97.5%": upper,
        "P_difference_above_0":
            np.mean(boot > 0)
    })


print(
    pd.DataFrame(
        bootstrap_results_offset
    ).to_string(index=False)
)

In [ ]:
final_17_comparison.to_csv(
    TABLE_DIR / "poisson_specification_selection.csv", index=False
)
pd.DataFrame(bootstrap_results_offset).to_csv(
    TABLE_DIR / "poisson_offset_vs_predictor_bootstrap.csv", index=False
)

## 5. Poisson recent-only robustness check

In [ ]:
# =========================================================
# Robustness check:
# Window selection for Poisson model WITHOUT LT information
#
# Candidate windows:
# 1, 3, 7, 10, 14, 21, 30 days
#
# Compare all single-, two-, and three-window combinations
# =========================================================

from itertools import combinations
from sklearn.linear_model import PoissonRegressor
import numpy as np
import pandas as pd


candidate_windows = [1, 3, 7, 10, 14, 21, 30]


def get_log_window(w, dates):
    return np.log1p(
        X_windows[w]
        .loc[dates]
        .to_numpy(dtype=float)
        .reshape(-1)
    )


# ---------------------------------------------------------
# Generate all 1-, 2-, and 3-window combinations
# ---------------------------------------------------------

candidate_sets = []

for r in [1, 2, 3]:
    candidate_sets.extend(
        combinations(candidate_windows, r)
    )


noLT_window_results = []
noLT_window_models = {}


# ---------------------------------------------------------
# Fit No-LT Poisson models
# ---------------------------------------------------------

for windows in candidate_sets:

    X_train_recent = np.column_stack([
        get_log_window(w, train_dates)
        for w in windows
    ])

    X_val_recent = np.column_stack([
        get_log_window(w, val_dates)
        for w in windows
    ])

    model = PoissonRegressor(
        alpha=0.0,
        fit_intercept=True,
        max_iter=3000,
        tol=1e-9
    )

    model.fit(
        X_train_recent,
        y_train
    )

    lambda_val = model.predict(
        X_val_recent
    )

    score = poisson_log_score(
        y_val,
        lambda_val
    )

    name = "+".join(
        str(w) for w in windows
    )

    noLT_window_results.append({
        "windows": name,
        "n_windows": len(windows),
        "validation_log_score": score,
        "iterations": model.n_iter_
    })

    noLT_window_models[name] = {
        "model": model,
        "lambda_val": lambda_val,
        "windows": windows
    }


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

noLT_window_comparison = (
    pd.DataFrame(noLT_window_results)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

print(
    noLT_window_comparison.to_string(
        index=False
    )
)

In [ ]:
# =========================================================
# Robustness comparison using the No-LT model's preferred
# windows: 7 + 10 + 30 days
# =========================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor


selected_windows = [7, 10, 30]


def build_selected_log_features(dates, windows):
    return np.column_stack([
        np.log1p(
            X_windows[w]
            .loc[dates]
            .to_numpy(dtype=float)
            .reshape(-1)
        )
        for w in windows
    ])


X_train_71030 = build_selected_log_features(
    train_dates,
    selected_windows
)

X_val_71030 = build_selected_log_features(
    val_dates,
    selected_windows
)

log_LT_train = np.log(LT_train)
log_LT_val = np.log(LT_val)


comparison_71030 = []
models_71030 = {}


# =========================================================
# 1. No LT
# =========================================================

model_noLT = PoissonRegressor(
    alpha=0.0,
    fit_intercept=True,
    max_iter=3000,
    tol=1e-9
)

model_noLT.fit(
    X_train_71030,
    y_train
)

lambda_noLT = model_noLT.predict(
    X_val_71030
)

comparison_71030.append({
    "model": "No LT",
    "windows": "7+10+30",
    "validation_log_score":
        poisson_log_score(y_val, lambda_noLT),
    "iterations": model_noLT.n_iter_
})

models_71030["No LT"] = {
    "model": model_noLT,
    "lambda_val": lambda_noLT
}


# =========================================================
# 2. LT as estimated predictor
# =========================================================

X_train_pred = np.column_stack([
    log_LT_train,
    X_train_71030
])

X_val_pred = np.column_stack([
    log_LT_val,
    X_val_71030
])

model_LTpred = PoissonRegressor(
    alpha=0.0,
    fit_intercept=True,
    max_iter=3000,
    tol=1e-9
)

model_LTpred.fit(
    X_train_pred,
    y_train
)

lambda_LTpred = model_LTpred.predict(
    X_val_pred
)

comparison_71030.append({
    "model": "LT predictor",
    "windows": "7+10+30",
    "validation_log_score":
        poisson_log_score(y_val, lambda_LTpred),
    "iterations": model_LTpred.n_iter_
})

models_71030["LT predictor"] = {
    "model": model_LTpred,
    "lambda_val": lambda_LTpred
}


# =========================================================
# 3. LT as fixed offset
# =========================================================

model_LToffset = PoissonRegressor(
    alpha=0.0,
    fit_intercept=True,
    max_iter=3000,
    tol=1e-9
)

model_LToffset.fit(
    X_train_71030,
    y_train / LT_train,
    sample_weight=LT_train
)

adjustment_val = model_LToffset.predict(
    X_val_71030
)

lambda_LToffset = (
    LT_val * adjustment_val
)

comparison_71030.append({
    "model": "LT offset",
    "windows": "7+10+30",
    "validation_log_score":
        poisson_log_score(y_val, lambda_LToffset),
    "iterations": model_LToffset.n_iter_
})

models_71030["LT offset"] = {
    "model": model_LToffset,
    "lambda_val": lambda_LToffset
}


# =========================================================
# Results
# =========================================================

comparison_71030 = (
    pd.DataFrame(comparison_71030)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

print(
    comparison_71030.to_string(index=False)
)


# Extra: LT coefficient in predictor model
print("\nLT-predictor coefficients:")
print("Intercept:", model_LTpred.intercept_)
print("beta_LT:", model_LTpred.coef_[0])

for w, coef in zip(
    selected_windows,
    model_LTpred.coef_[1:]
):
    print(f"beta_{w}:", coef)

## 6. XGBoost long-term-information comparison

In [ ]:
# =========================================================
# XGBoost experiment 1:
# How should long-term spatial information enter the model?
# =========================================================

import numpy as np
import pandas as pd
import xgboost as xgb

print("XGBoost version:", xgb.__version__)


candidate_windows = [1, 3, 7, 10, 14, 21, 30]


def build_xgb_recent_features(dates):
    """
    Raw recent-seismicity counts for all candidate windows.
    Rows follow the same day-cell order as y_train / y_val.
    """
    return np.column_stack([
        X_windows[w]
        .loc[dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1)
        for w in candidate_windows
    ]).astype(np.float32)


X_train_recent_xgb = build_xgb_recent_features(train_dates)
X_val_recent_xgb   = build_xgb_recent_features(val_dates)

feature_names_recent = [
    f"X{w}" for w in candidate_windows
]

log_LT_train_xgb = np.log(
    LT_train.astype(np.float64)
).astype(np.float32)

log_LT_val_xgb = np.log(
    LT_val.astype(np.float64)
).astype(np.float32)


print("Training feature shape:", X_train_recent_xgb.shape)
print("Validation feature shape:", X_val_recent_xgb.shape)

print("Training target shape:", y_train.shape)
print("Validation target shape:", y_val.shape)

print("LT margin range:",
      log_LT_train_xgb.min(),
      log_LT_train_xgb.max())

In [ ]:
# =========================================================
# Common provisional parameters
# These are NOT the final tuned hyperparameters.
# =========================================================

xgb_params_screen = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",

    # Moderate initial tree complexity
    "max_depth": 4,
    "eta": 0.05,

    # Keep the first comparison deterministic and comparable
    "subsample": 1.0,
    "colsample_bytree": 1.0,

    "min_child_weight": 1,
    "lambda": 1.0,
    "alpha": 0.0,

    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}

In [ ]:
# =========================================================
# A. No LT
# =========================================================

dtrain_noLT = xgb.DMatrix(
    X_train_recent_xgb,
    label=y_train,
    feature_names=feature_names_recent
)

dval_noLT = xgb.DMatrix(
    X_val_recent_xgb,
    label=y_val,
    feature_names=feature_names_recent
)

model_noLT_xgb = xgb.train(
    params=xgb_params_screen,
    dtrain=dtrain_noLT,
    num_boost_round=2000,
    evals=[
        (dtrain_noLT, "train"),
        (dval_noLT, "validation")
    ],
    early_stopping_rounds=100,
    verbose_eval=100
)

lambda_noLT_xgb = model_noLT_xgb.predict(
    dval_noLT,
    iteration_range=(
        0,
        model_noLT_xgb.best_iteration + 1
    )
)

In [ ]:
# =========================================================
# B. LT as ordinary feature
# =========================================================

X_train_LTfeature = np.column_stack([
    log_LT_train_xgb,
    X_train_recent_xgb
]).astype(np.float32)

X_val_LTfeature = np.column_stack([
    log_LT_val_xgb,
    X_val_recent_xgb
]).astype(np.float32)

feature_names_LT = [
    "log_LT"
] + feature_names_recent


dtrain_LTfeature = xgb.DMatrix(
    X_train_LTfeature,
    label=y_train,
    feature_names=feature_names_LT
)

dval_LTfeature = xgb.DMatrix(
    X_val_LTfeature,
    label=y_val,
    feature_names=feature_names_LT
)

model_LTfeature_xgb = xgb.train(
    params=xgb_params_screen,
    dtrain=dtrain_LTfeature,
    num_boost_round=2000,
    evals=[
        (dtrain_LTfeature, "train"),
        (dval_LTfeature, "validation")
    ],
    early_stopping_rounds=100,
    verbose_eval=100
)

lambda_LTfeature_xgb = model_LTfeature_xgb.predict(
    dval_LTfeature,
    iteration_range=(
        0,
        model_LTfeature_xgb.best_iteration + 1
    )
)

In [ ]:
# =========================================================
# C. LT as base margin
# =========================================================

dtrain_LTmargin = xgb.DMatrix(
    X_train_recent_xgb,
    label=y_train,
    base_margin=log_LT_train_xgb,
    feature_names=feature_names_recent
)

dval_LTmargin = xgb.DMatrix(
    X_val_recent_xgb,
    label=y_val,
    base_margin=log_LT_val_xgb,
    feature_names=feature_names_recent
)

model_LTmargin_xgb = xgb.train(
    params=xgb_params_screen,
    dtrain=dtrain_LTmargin,
    num_boost_round=2000,
    evals=[
        (dtrain_LTmargin, "train"),
        (dval_LTmargin, "validation")
    ],
    early_stopping_rounds=100,
    verbose_eval=100
)

lambda_LTmargin_xgb = model_LTmargin_xgb.predict(
    dval_LTmargin,
    iteration_range=(
        0,
        model_LTmargin_xgb.best_iteration + 1
    )
)

In [ ]:
# =========================================================
# D. LT as base margin + LT as feature
# =========================================================

dtrain_both = xgb.DMatrix(
    X_train_LTfeature,
    label=y_train,
    base_margin=log_LT_train_xgb,
    feature_names=feature_names_LT
)

dval_both = xgb.DMatrix(
    X_val_LTfeature,
    label=y_val,
    base_margin=log_LT_val_xgb,
    feature_names=feature_names_LT
)

model_both_xgb = xgb.train(
    params=xgb_params_screen,
    dtrain=dtrain_both,
    num_boost_round=2000,
    evals=[
        (dtrain_both, "train"),
        (dval_both, "validation")
    ],
    early_stopping_rounds=100,
    verbose_eval=100
)

lambda_both_xgb = model_both_xgb.predict(
    dval_both,
    iteration_range=(
        0,
        model_both_xgb.best_iteration + 1
    )
)

In [ ]:
# =========================================================
# Statewide mean forecast for each model
# =========================================================

for name, pred in {
    "No LT": lambda_noLT_xgb,
    "LT as feature": lambda_LTfeature_xgb,
    "LT as base margin": lambda_LTmargin_xgb,
    "LT margin + feature": lambda_both_xgb
}.items():

    pred_matrix = pred.reshape(
        len(val_dates),
        len(cell_ids)
    )

    statewide_mean = (
        pred_matrix.sum(axis=1).mean()
    )

    print(
        name,
        "mean statewide 7-day forecast =",
        statewide_mean
    )

In [ ]:
models_check = {
    "No LT": model_noLT_xgb,
    "LT as feature": model_LTfeature_xgb,
    "LT as base margin": model_LTmargin_xgb,
    "LT margin + feature": model_both_xgb
}

predictions_check = {
    "No LT": lambda_noLT_xgb,
    "LT as feature": lambda_LTfeature_xgb,
    "LT as base margin": lambda_LTmargin_xgb,
    "LT margin + feature": lambda_both_xgb
}

rows = []

for name, model in models_check.items():
    rows.append({
        "model": name,
        "best_iteration": model.best_iteration + 1,
        "best_validation_nloglik": model.best_score,
        "validation_log_score":
            poisson_log_score(
                y_val,
                predictions_check[name]
            )
    })

pd.DataFrame(rows).sort_values(
    "validation_log_score",
    ascending=False
)

In [ ]:
# =========================================================
# XGBoost LT-treatment robustness check
# Vary tree complexity under the same search budget
# =========================================================

import itertools
import pandas as pd
import xgboost as xgb


# ---------------------------------------------------------
# Four LT treatments already constructed above
# ---------------------------------------------------------

xgb_datasets = {
    "No LT": (dtrain_noLT, dval_noLT),
    "LT as feature": (dtrain_LTfeature, dval_LTfeature),
    "LT as base margin": (dtrain_LTmargin, dval_LTmargin),
    "LT margin + feature": (dtrain_both, dval_both)
}


# ---------------------------------------------------------
# Parameter grid
# ---------------------------------------------------------

max_depth_grid = [2, 4, 6]
min_child_weight_grid = [1, 10, 50]


base_params = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",

    # Fixed during this experiment
    "eta": 0.05,
    "subsample": 1.0,
    "colsample_bytree": 1.0,

    "lambda": 1.0,
    "alpha": 0.0,

    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}


# ---------------------------------------------------------
# Run all 36 models
# ---------------------------------------------------------

results = []

total_fits = (
    len(xgb_datasets)
    * len(max_depth_grid)
    * len(min_child_weight_grid)
)

fit_no = 0


for model_name, (dtrain, dval) in xgb_datasets.items():

    for max_depth, min_child_weight in itertools.product(
        max_depth_grid,
        min_child_weight_grid
    ):

        fit_no += 1

        print(
            f"[{fit_no}/{total_fits}] "
            f"{model_name} | "
            f"depth={max_depth}, "
            f"min_child_weight={min_child_weight}"
        )

        params = base_params.copy()

        params.update({
            "max_depth": max_depth,
            "min_child_weight": min_child_weight
        })

        model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=2000,
            evals=[(dval, "validation")],
            early_stopping_rounds=100,
            verbose_eval=False
        )

        best_nloglik = float(model.best_score)

        results.append({
            "model": model_name,
            "max_depth": max_depth,
            "min_child_weight": min_child_weight,
            "best_iteration": model.best_iteration + 1,
            "validation_nloglik": best_nloglik,
            "validation_log_score": -best_nloglik
        })


xgb_tuning_results = pd.DataFrame(results)

print("\nFinished.")

In [ ]:
# =========================================================
# 1. Best result for each LT treatment
# =========================================================

best_by_LT = (
    xgb_tuning_results
    .sort_values(
        "validation_log_score",
        ascending=False
    )
    .groupby("model", as_index=False)
    .first()
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

best_by_LT

In [ ]:
# =========================================================
# 2. Compare LT treatments under every parameter combination
# =========================================================

comparison_grid = (
    xgb_tuning_results
    .pivot_table(
        index=["max_depth", "min_child_weight"],
        columns="model",
        values="validation_log_score"
    )
    .reset_index()
)

comparison_grid

In [ ]:
# =========================================================
# 3. Which LT treatment wins under each parameter setting?
# =========================================================

winner_by_setting = (
    xgb_tuning_results
    .loc[
        xgb_tuning_results
        .groupby(["max_depth", "min_child_weight"])
        ["validation_log_score"]
        .idxmax()
    ]
    [
        [
            "max_depth",
            "min_child_weight",
            "model",
            "validation_log_score",
            "best_iteration"
        ]
    ]
    .sort_values(
        ["max_depth", "min_child_weight"]
    )
)

winner_by_setting

In [ ]:
winner_by_setting["model"].value_counts()

## 7. XGBoost backward-elimination screening

In [ ]:
# ============================================================
# XGBoost recent-seismicity window selection
# Greedy backward elimination
#
# Fixed throughout this experiment:
#   - objective = count:poisson
#   - LT = ordinary feature
#   - max_depth = 4
#   - min_child_weight = 1
#   - eta = 0.05
#
# Only the recent-seismicity window set changes.
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import time


# ------------------------------------------------------------
# 1. Candidate windows
# ------------------------------------------------------------

candidate_windows = [1, 3, 7, 10, 14, 21, 30]


# ------------------------------------------------------------
# 2. Fixed XGBoost settings for WINDOW SELECTION
#
# Important:
# These are provisional settings used to control other factors
# while selecting windows.
# They are NOT yet the final tuned XGBoost hyperparameters.
# ------------------------------------------------------------

window_selection_params = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",

    "max_depth": 4,
    "min_child_weight": 1,
    "eta": 0.05,

    "subsample": 1.0,
    "colsample_bytree": 1.0,

    "lambda": 1.0,
    "alpha": 0.0,

    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}


# ------------------------------------------------------------
# 3. Function to fit one window subset
#
# LT is ALWAYS retained as a feature.
# Only recent-seismicity windows change.
# ------------------------------------------------------------

def fit_xgb_window_subset(windows, verbose=False):

    windows = list(windows)

    # Column positions in X_train_recent_xgb / X_val_recent_xgb
    cols = [
        candidate_windows.index(w)
        for w in windows
    ]

    # Add log_LT as the first feature
    Xtr = np.column_stack([
        log_LT_train_xgb,
        X_train_recent_xgb[:, cols]
    ]).astype(np.float32)

    Xva = np.column_stack([
        log_LT_val_xgb,
        X_val_recent_xgb[:, cols]
    ]).astype(np.float32)

    feature_names = (
        ["log_LT"]
        + [f"X{w}" for w in windows]
    )

    dtrain = xgb.DMatrix(
        Xtr,
        label=y_train,
        feature_names=feature_names
    )

    dval = xgb.DMatrix(
        Xva,
        label=y_val,
        feature_names=feature_names
    )

    model = xgb.train(
        params=window_selection_params,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=[(dval, "validation")],
        early_stopping_rounds=100,
        verbose_eval=verbose
    )

    validation_nloglik = float(model.best_score)

    result = {
        "windows": tuple(windows),
        "n_windows": len(windows),
        "validation_nloglik": validation_nloglik,
        "validation_log_score": -validation_nloglik,
        "best_iteration": model.best_iteration + 1
    }

    # Free large matrices immediately
    del Xtr, Xva, dtrain, dval, model
    gc.collect()

    return result

In [ ]:
# ============================================================
# 4. Run complete backward-elimination path
# ============================================================

current_windows = candidate_windows.copy()

all_step_results = []
backward_path = []

start_time = time.time()


# ------------------------------------------------------------
# First fit: all 7 windows
# ------------------------------------------------------------

print("=" * 70)
print("Initial model: all candidate windows")
print("=" * 70)

current_result = fit_xgb_window_subset(
    current_windows
)

backward_path.append({
    "step": 0,
    "n_windows": len(current_windows),
    "windows": "+".join(map(str, current_windows)),
    "dropped_window": None,
    "validation_log_score":
        current_result["validation_log_score"],
    "change_from_previous": np.nan,
    "best_iteration":
        current_result["best_iteration"]
})

print(
    f"Windows: {current_windows}\n"
    f"Validation log score: "
    f"{current_result['validation_log_score']:.9f}\n"
    f"Best iteration: "
    f"{current_result['best_iteration']}"
)


# ------------------------------------------------------------
# Repeatedly remove one window
# ------------------------------------------------------------

step = 1

while len(current_windows) > 1:

    print("\n" + "=" * 70)
    print(
        f"STEP {step}: "
        f"{len(current_windows)} -> "
        f"{len(current_windows)-1} windows"
    )
    print("=" * 70)

    step_candidates = []

    current_score = (
        backward_path[-1]["validation_log_score"]
    )

    # Try deleting every remaining window
    for dropped_window in current_windows:

        remaining_windows = [
            w for w in current_windows
            if w != dropped_window
        ]

        print(
            f"Testing drop X{dropped_window:<2} | "
            f"Remaining: {remaining_windows}"
        )

        result = fit_xgb_window_subset(
            remaining_windows
        )

        row = {
            "step": step,
            "starting_windows":
                "+".join(map(str, current_windows)),
            "dropped_window": dropped_window,
            "remaining_windows":
                "+".join(map(str, remaining_windows)),
            "n_windows":
                len(remaining_windows),
            "validation_log_score":
                result["validation_log_score"],
            "difference_vs_current":
                result["validation_log_score"]
                - current_score,
            "best_iteration":
                result["best_iteration"]
        }

        step_candidates.append(row)
        all_step_results.append(row)

        print(
            f"    score = "
            f"{result['validation_log_score']:.9f} | "
            f"Δ = "
            f"{row['difference_vs_current']:+.9f} | "
            f"iter = "
            f"{result['best_iteration']}"
        )

    # --------------------------------------------------------
    # Select the best one-window deletion at this step
    # --------------------------------------------------------

    step_df = pd.DataFrame(step_candidates)

    best_row = (
        step_df
        .sort_values(
            "validation_log_score",
            ascending=False
        )
        .iloc[0]
    )

    dropped = int(best_row["dropped_window"])

    new_windows = [
        w for w in current_windows
        if w != dropped
    ]

    print("\nBest deletion this step:")
    print(
        f"Drop X{dropped} -> "
        f"{new_windows}"
    )
    print(
        f"Validation log score = "
        f"{best_row['validation_log_score']:.9f}"
    )
    print(
        f"Change from previous model = "
        f"{best_row['difference_vs_current']:+.9f}"
    )

    backward_path.append({
        "step": step,
        "n_windows": len(new_windows),
        "windows":
            "+".join(map(str, new_windows)),
        "dropped_window": dropped,
        "validation_log_score":
            best_row["validation_log_score"],
        "change_from_previous":
            best_row["difference_vs_current"],
        "best_iteration":
            int(best_row["best_iteration"])
    })

    current_windows = new_windows

    step += 1


elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("Backward elimination complete")
print(
    f"Total elapsed time: "
    f"{elapsed / 60:.1f} minutes"
)
print("=" * 70)

In [ ]:
# ============================================================
# 5. Summary of the selected backward path
# ============================================================

backward_path_df = pd.DataFrame(
    backward_path
)

backward_path_df

In [ ]:
# ============================================================
# 8. Numerically best model along the backward path
# ============================================================

best_path_model = (
    backward_path_df
    .sort_values(
        "validation_log_score",
        ascending=False
    )
    .iloc[0]
)

best_path_model

In [ ]:
# ============================================================
# 9. Simple diagnostic for the stopping point
# ============================================================

stop_diagnostic = backward_path_df.copy()

stop_diagnostic["effect_of_deletion"] = np.where(
    stop_diagnostic["change_from_previous"].isna(),
    "Starting model",
    np.where(
        stop_diagnostic["change_from_previous"] > 0,
        "Improved",
        "Worse"
    )
)

stop_diagnostic[
    [
        "n_windows",
        "windows",
        "dropped_window",
        "validation_log_score",
        "change_from_previous",
        "effect_of_deletion"
    ]
]

## 8. Matched XGBoost hyperparameter search

In [ ]:
# ============================================================
# XGBoost final hyperparameter tuning
# Same search configurations for all feature sets
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import itertools
import gc
import time


# ------------------------------------------------------------
# Feature sets to compare
# ------------------------------------------------------------

feature_sets = {
    "7+30": [7, 30],
    "1+7+30": [1, 7, 30],
    "all7": [1, 3, 7, 10, 14, 21, 30]
}


# ------------------------------------------------------------
# Hyperparameter candidate values
# ------------------------------------------------------------

search_space = {
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [0.5, 1, 5, 10, 50],
    "subsample": [0.70, 0.85, 1.00],
    "colsample_bytree": [0.70, 0.85, 1.00],
    "reg_lambda": [0.1, 1.0, 5.0, 10.0],
    "reg_alpha": [0.0, 0.1, 1.0]
}


# ------------------------------------------------------------
# Construct complete grid
# ------------------------------------------------------------

all_configs = list(
    itertools.product(
        search_space["max_depth"],
        search_space["min_child_weight"],
        search_space["subsample"],
        search_space["colsample_bytree"],
        search_space["reg_lambda"],
        search_space["reg_alpha"]
    )
)

print("Total possible configurations:", len(all_configs))

In [ ]:
# ============================================================
# Reproducible random sample of configurations
# ============================================================

rng = np.random.default_rng(2026)

n_random_configs = 40

selected_indices = rng.choice(
    len(all_configs),
    size=n_random_configs,
    replace=False
)

selected_configs = [
    all_configs[i]
    for i in selected_indices
]


def config_to_dict(config):
    return {
        "max_depth": int(config[0]),
        "min_child_weight": float(config[1]),
        "subsample": float(config[2]),
        "colsample_bytree": float(config[3]),
        "reg_lambda": float(config[4]),
        "reg_alpha": float(config[5])
    }


selected_configs = [
    config_to_dict(c)
    for c in selected_configs
]

In [ ]:
baseline_config = {
    "max_depth": 4,
    "min_child_weight": 1.0,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0
}

if baseline_config not in selected_configs:
    selected_configs.append(baseline_config)

print(
    "Number of configurations used:",
    len(selected_configs)
)

In [ ]:
# ============================================================
# Build DMatrix for one recent-window specification
# ============================================================

def make_xgb_dmatrices(windows):

    cols = [
        candidate_windows.index(w)
        for w in windows
    ]

    # LT always included as ordinary feature
    Xtr = np.column_stack([
        log_LT_train_xgb,
        X_train_recent_xgb[:, cols]
    ]).astype(np.float32)

    Xva = np.column_stack([
        log_LT_val_xgb,
        X_val_recent_xgb[:, cols]
    ]).astype(np.float32)

    feature_names = (
        ["log_LT"]
        + [f"X{w}" for w in windows]
    )

    dtrain = xgb.DMatrix(
        Xtr,
        label=y_train,
        feature_names=feature_names
    )

    dval = xgb.DMatrix(
        Xva,
        label=y_val,
        feature_names=feature_names
    )

    return dtrain, dval

In [ ]:
# ============================================================
# Run identical hyperparameter search for all feature sets
# ============================================================

tuning_results = []

total_fits = (
    len(feature_sets)
    * len(selected_configs)
)

fit_no = 0
start_time = time.time()


for feature_set_name, windows in feature_sets.items():

    print("\n" + "=" * 75)
    print(
        f"FEATURE SET: {feature_set_name} "
        f"| windows = {windows}"
    )
    print("=" * 75)

    dtrain, dval = make_xgb_dmatrices(
        windows
    )

    for config_id, config in enumerate(
        selected_configs,
        start=1
    ):

        fit_no += 1

        print(
            f"[{fit_no}/{total_fits}] "
            f"{feature_set_name} | "
            f"depth={config['max_depth']} | "
            f"child={config['min_child_weight']} | "
            f"sub={config['subsample']} | "
            f"col={config['colsample_bytree']} | "
            f"L2={config['reg_lambda']} | "
            f"L1={config['reg_alpha']}"
        )

        params = {
            "objective": "count:poisson",
            "eval_metric": "poisson-nloglik",

            # Fixed learning rate
            "eta": 0.05,

            # Tuned parameters
            **config,

            "tree_method": "hist",
            "seed": 2026,
            "nthread": -1
        }

        model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=2000,
            evals=[
                (dval, "validation")
            ],
            early_stopping_rounds=100,
            verbose_eval=False
        )

        tuning_results.append({
            "feature_set": feature_set_name,
            "windows": "+".join(
                map(str, windows)
            ),
            "config_id": config_id,

            **config,

            "best_iteration":
                model.best_iteration + 1,

            "validation_nloglik":
                float(model.best_score),

            "validation_log_score":
                -float(model.best_score)
        })

        del model
        gc.collect()

    del dtrain, dval
    gc.collect()


elapsed = time.time() - start_time

print("\n" + "=" * 75)
print("TUNING COMPLETE")
print(
    f"Elapsed time: "
    f"{elapsed / 60:.1f} minutes"
)
print("=" * 75)


xgb_tuning_results = pd.DataFrame(
    tuning_results
)

In [ ]:
# ============================================================
# Best configuration for each feature set
# ============================================================

best_by_feature_set = (
    xgb_tuning_results
    .sort_values(
        "validation_log_score",
        ascending=False
    )
    .groupby(
        "feature_set",
        as_index=False
    )
    .first()
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

best_by_feature_set[
    [
        "feature_set",
        "windows",
        "validation_log_score",
        "max_depth",
        "min_child_weight",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
        "reg_alpha",
        "best_iteration"
    ]
]

In [ ]:
# ============================================================
# Overall best tuning results
# ============================================================

top15 = (
    xgb_tuning_results
    .sort_values(
        "validation_log_score",
        ascending=False
    )
    .head(15)
)

top15[
    [
        "feature_set",
        "validation_log_score",
        "max_depth",
        "min_child_weight",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
        "reg_alpha",
        "best_iteration"
    ]
]

In [ ]:
# ============================================================
# Winner under each matched hyperparameter configuration
# ============================================================

winner_by_config = (
    xgb_tuning_results
    .loc[
        xgb_tuning_results
        .groupby("config_id")
        ["validation_log_score"]
        .idxmax()
    ]
    [
        [
            "config_id",
            "feature_set",
            "validation_log_score"
        ]
    ]
)

winner_counts = (
    winner_by_config["feature_set"]
    .value_counts()
)

winner_counts

In [ ]:
# ============================================================
# Additional tuning:
# 5-window and 4-window candidates
# Use EXACTLY the same 41 configurations as before
# ============================================================

additional_feature_sets = {
    "1+7+10+21+30": [1, 7, 10, 21, 30],
    "1+7+21+30": [1, 7, 21, 30]
}

additional_results = []

total_fits = (
    len(additional_feature_sets)
    * len(selected_configs)
)

fit_no = 0

for feature_set_name, windows in additional_feature_sets.items():

    print("\n" + "=" * 75)
    print(
        f"FEATURE SET: {feature_set_name} "
        f"| windows = {windows}"
    )
    print("=" * 75)

    dtrain, dval = make_xgb_dmatrices(windows)

    for config_id, config in enumerate(
        selected_configs,
        start=1
    ):

        fit_no += 1

        print(
            f"[{fit_no}/{total_fits}] "
            f"{feature_set_name} | "
            f"depth={config['max_depth']} | "
            f"child={config['min_child_weight']} | "
            f"sub={config['subsample']} | "
            f"col={config['colsample_bytree']} | "
            f"L2={config['reg_lambda']} | "
            f"L1={config['reg_alpha']}"
        )

        params = {
            "objective": "count:poisson",
            "eval_metric": "poisson-nloglik",
            "eta": 0.05,

            **config,

            "tree_method": "hist",
            "seed": 2026,
            "nthread": -1
        }

        model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=2000,
            evals=[(dval, "validation")],
            early_stopping_rounds=100,
            verbose_eval=False
        )

        additional_results.append({
            "feature_set": feature_set_name,
            "windows": "+".join(map(str, windows)),
            "config_id": config_id,

            **config,

            "best_iteration":
                model.best_iteration + 1,

            "validation_nloglik":
                float(model.best_score),

            "validation_log_score":
                -float(model.best_score)
        })

        del model
        gc.collect()

    del dtrain, dval
    gc.collect()

In [ ]:
additional_results_df = pd.DataFrame(
    additional_results
)

xgb_tuning_results_complete = pd.concat(
    [
        xgb_tuning_results,
        additional_results_df
    ],
    ignore_index=True
)

In [ ]:
best_complete = (
    xgb_tuning_results_complete
    .sort_values(
        "validation_log_score",
        ascending=False
    )
    .groupby(
        "feature_set",
        as_index=False
    )
    .first()
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

best_complete[
    [
        "feature_set",
        "validation_log_score",
        "max_depth",
        "min_child_weight",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
        "reg_alpha",
        "best_iteration"
    ]
]

In [ ]:
winner_complete = (
    xgb_tuning_results_complete
    .loc[
        xgb_tuning_results_complete
        .groupby("config_id")
        ["validation_log_score"]
        .idxmax()
    ]
)

winner_complete["feature_set"].value_counts()

## 9. Local sensitivity around 21 days

In [ ]:
# ============================================================
# Local sensitivity check around the 21-day window
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import gc

local_windows = [18, 19, 20, 21, 22, 23, 24]

# X_windows[1] is the daily M>=2.5 count matrix C_{t,g}
daily_counts_from_X1 = X_windows[1].copy()

X_local = {}

for w in local_windows:
    X_local[w] = (
        daily_counts_from_X1
        .rolling(window=w, min_periods=w)
        .sum()
    )

print("Local windows constructed:", list(X_local.keys()))

In [ ]:
check_dates = train_dates.union(val_dates)

difference_21 = (
    X_local[21]
    .loc[check_dates]
    .to_numpy()
    -
    X_windows[21]
    .loc[check_dates]
    .to_numpy()
)

print(
    "Maximum absolute difference for reconstructed X21:",
    np.nanmax(np.abs(difference_21))
)

In [ ]:
local_window_params = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",

    "eta": 0.05,

    "max_depth": 4,
    "min_child_weight": 0.5,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_lambda": 10.0,
    "reg_alpha": 1.0,

    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}

In [ ]:
# ============================================================
# Fit 1 + 7 + w + 30 for w = 18,...,24
# ============================================================

local_results = []

for w in local_windows:

    print(
        f"Testing windows: [1, 7, {w}, 30]"
    )

    # Training features
    Xtr = np.column_stack([
        log_LT_train_xgb,

        X_windows[1]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[7]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_local[w]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[30]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1)
    ]).astype(np.float32)


    # Validation features
    Xva = np.column_stack([
        log_LT_val_xgb,

        X_windows[1]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[7]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_local[w]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[30]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1)
    ]).astype(np.float32)


    feature_names = [
        "log_LT",
        "X1",
        "X7",
        f"X{w}",
        "X30"
    ]

    dtrain = xgb.DMatrix(
        Xtr,
        label=y_train,
        feature_names=feature_names
    )

    dval = xgb.DMatrix(
        Xva,
        label=y_val,
        feature_names=feature_names
    )

    model = xgb.train(
        params=local_window_params,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=[(dval, "validation")],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    local_results.append({
        "middle_window": w,
        "windows": f"1+7+{w}+30",
        "validation_log_score":
            -float(model.best_score),
        "best_iteration":
            model.best_iteration + 1
    })

    del Xtr, Xva, dtrain, dval, model
    gc.collect()

In [ ]:
local_window_results = (
    pd.DataFrame(local_results)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

local_window_results

In [ ]:
score_21 = (
    local_window_results
    .loc[
        local_window_results["middle_window"] == 21,
        "validation_log_score"
    ]
    .iloc[0]
)

local_window_results["difference_vs_21"] = (
    local_window_results["validation_log_score"]
    - score_21
)

local_window_results[
    [
        "middle_window",
        "windows",
        "validation_log_score",
        "difference_vs_21",
        "best_iteration"
    ]
]

In [ ]:
# ============================================================
# Extend local sensitivity check: 25-29 days
# ============================================================

extra_local_windows = [25, 26, 27, 28, 29]

for w in extra_local_windows:
    X_local[w] = (
        daily_counts_from_X1
        .rolling(window=w, min_periods=w)
        .sum()
    )

extra_results = []

for w in extra_local_windows:

    print(f"Testing windows: [1, 7, {w}, 30]")

    Xtr = np.column_stack([
        log_LT_train_xgb,

        X_windows[1]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[7]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_local[w]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[30]
        .loc[train_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1)
    ]).astype(np.float32)

    Xva = np.column_stack([
        log_LT_val_xgb,

        X_windows[1]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[7]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_local[w]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1),

        X_windows[30]
        .loc[val_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1)
    ]).astype(np.float32)

    feature_names = [
        "log_LT",
        "X1",
        "X7",
        f"X{w}",
        "X30"
    ]

    dtrain = xgb.DMatrix(
        Xtr,
        label=y_train,
        feature_names=feature_names
    )

    dval = xgb.DMatrix(
        Xva,
        label=y_val,
        feature_names=feature_names
    )

    model = xgb.train(
        params=local_window_params,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=[(dval, "validation")],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    extra_results.append({
        "middle_window": w,
        "windows": f"1+7+{w}+30",
        "validation_log_score": -float(model.best_score),
        "best_iteration": model.best_iteration + 1
    })

    del Xtr, Xva, dtrain, dval, model
    gc.collect()

In [ ]:
extra_local_results = pd.DataFrame(extra_results)

local_window_results_full = pd.concat(
    [
        local_window_results.drop(
            columns=["difference_vs_21"],
            errors="ignore"
        ),
        extra_local_results
    ],
    ignore_index=True
)

score_21 = (
    local_window_results_full
    .loc[
        local_window_results_full["middle_window"] == 21,
        "validation_log_score"
    ]
    .iloc[0]
)

local_window_results_full["difference_vs_21"] = (
    local_window_results_full["validation_log_score"]
    - score_21
)

local_window_results_full.sort_values(
    "middle_window"
)[
    [
        "middle_window",
        "validation_log_score",
        "difference_vs_21",
        "best_iteration"
    ]
]

## 10. Final XGBoost local refinement

In [ ]:
# ============================================================
# Final XGBoost feature set for local refinement
# LT + 1, 7, 21, 30-day recent seismicity
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import gc

final_windows = [1, 7, 21, 30]

cols = [
    candidate_windows.index(w)
    for w in final_windows
]

Xtr_final = np.column_stack([
    log_LT_train_xgb,
    X_train_recent_xgb[:, cols]
]).astype(np.float32)

Xva_final = np.column_stack([
    log_LT_val_xgb,
    X_val_recent_xgb[:, cols]
]).astype(np.float32)

final_feature_names = [
    "log_LT",
    "X1",
    "X7",
    "X21",
    "X30"
]

dtrain_final = xgb.DMatrix(
    Xtr_final,
    label=y_train,
    feature_names=final_feature_names
)

dval_final = xgb.DMatrix(
    Xva_final,
    label=y_val,
    feature_names=final_feature_names
)

print("Training shape:", Xtr_final.shape)
print("Validation shape:", Xva_final.shape)

In [ ]:
# ============================================================
# Current best configuration
# ============================================================

current_best = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",

    "eta": 0.05,

    "max_depth": 4,
    "min_child_weight": 0.5,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_lambda": 10.0,
    "reg_alpha": 1.0,

    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}

In [ ]:
def fit_refinement_model(params):

    model = xgb.train(
        params=params,
        dtrain=dtrain_final,
        num_boost_round=2000,
        evals=[(dval_final, "validation")],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    return {
        "validation_log_score":
            -float(model.best_score),
        "best_iteration":
            model.best_iteration + 1
    }

In [ ]:
# ============================================================
# A. min_child_weight boundary check
# ============================================================

child_values = [
    0.0,
    0.1,
    0.25,
    0.5,
    1.0,
    2.0
]

child_results = []

for value in child_values:

    params = current_best.copy()
    params["min_child_weight"] = value

    result = fit_refinement_model(params)

    child_results.append({
        "min_child_weight": value,
        **result
    })

child_results_df = (
    pd.DataFrame(child_results)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

child_results_df

In [ ]:
# ============================================================
# B. L2 regularisation boundary check
# ============================================================

lambda_values = [
    2.0,
    5.0,
    10.0,
    20.0,
    50.0
]

lambda_results = []

for value in lambda_values:

    params = current_best.copy()
    params["reg_lambda"] = value

    result = fit_refinement_model(params)

    lambda_results.append({
        "reg_lambda": value,
        **result
    })

lambda_results_df = (
    pd.DataFrame(lambda_results)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

lambda_results_df

In [ ]:
# ============================================================
# C. L1 regularisation boundary check
# ============================================================

alpha_values = [
    0.0,
    0.25,
    0.5,
    1.0,
    2.0,
    5.0
]

alpha_results = []

for value in alpha_values:

    params = current_best.copy()
    params["reg_alpha"] = value

    result = fit_refinement_model(params)

    alpha_results.append({
        "reg_alpha": value,
        **result
    })

alpha_results_df = (
    pd.DataFrame(alpha_results)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

alpha_results_df

In [ ]:
# ============================================================
# Extra L2 check below lambda = 2
# Everything else stays at the current reference configuration
# ============================================================

lambda_values_lower = [
    0.0,
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    3.0,
    5.0
]

lambda_lower_results = []

for value in lambda_values_lower:

    params = current_best.copy()

    # Use the better values suggested by the one-at-a-time checks
    params["min_child_weight"] = 1.0
    params["reg_alpha"] = 0.0
    params["reg_lambda"] = value

    result = fit_refinement_model(params)

    lambda_lower_results.append({
        "reg_lambda": value,
        **result
    })

lambda_lower_results_df = (
    pd.DataFrame(lambda_lower_results)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

lambda_lower_results_df

In [ ]:
# ============================================================
# Final joint local refinement
# Final feature set: LT + 1, 7, 21, 30 days
# ============================================================

import itertools
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import time


# ------------------------------------------------------------
# Local search space
# ------------------------------------------------------------

local_search_space = {
    "max_depth": [3, 4, 5],
    "min_child_weight": [0.5, 1.0, 2.0],
    "subsample": [0.75, 0.85, 0.95],
    "colsample_bytree": [0.75, 0.85, 0.95],
    "reg_lambda": [0.1, 0.25, 0.5, 1.0],
    "reg_alpha": [0.0, 0.1, 0.25]
}


all_local_configs = list(
    itertools.product(
        local_search_space["max_depth"],
        local_search_space["min_child_weight"],
        local_search_space["subsample"],
        local_search_space["colsample_bytree"],
        local_search_space["reg_lambda"],
        local_search_space["reg_alpha"]
    )
)

print("Total local configurations:", len(all_local_configs))

In [ ]:
# ============================================================
# Reproducible random sample
# ============================================================

rng = np.random.default_rng(2026)

n_local_random = 60

selected_idx = rng.choice(
    len(all_local_configs),
    size=n_local_random,
    replace=False
)


def local_config_to_dict(c):
    return {
        "max_depth": int(c[0]),
        "min_child_weight": float(c[1]),
        "subsample": float(c[2]),
        "colsample_bytree": float(c[3]),
        "reg_lambda": float(c[4]),
        "reg_alpha": float(c[5])
    }


local_configs = [
    local_config_to_dict(all_local_configs[i])
    for i in selected_idx
]

In [ ]:
current_local_best = {
    "max_depth": 4,
    "min_child_weight": 1.0,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 0.25,
    "reg_alpha": 0.0
}

if current_local_best not in local_configs:
    local_configs.append(current_local_best)

print(
    "Configurations to fit:",
    len(local_configs)
)

In [ ]:
# ============================================================
# Run joint local refinement
# ============================================================

joint_results = []

start_time = time.time()

for i, config in enumerate(local_configs, start=1):

    print(
        f"[{i}/{len(local_configs)}] "
        f"depth={config['max_depth']} | "
        f"child={config['min_child_weight']} | "
        f"sub={config['subsample']} | "
        f"col={config['colsample_bytree']} | "
        f"L2={config['reg_lambda']} | "
        f"L1={config['reg_alpha']}"
    )

    params = {
        "objective": "count:poisson",
        "eval_metric": "poisson-nloglik",

        # Still fixed for this stage
        "eta": 0.05,

        **config,

        "tree_method": "hist",
        "seed": 2026,
        "nthread": -1
    }

    model = xgb.train(
        params=params,
        dtrain=dtrain_final,
        num_boost_round=2500,
        evals=[(dval_final, "validation")],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    joint_results.append({
        **config,

        "eta": 0.05,

        "validation_nloglik":
            float(model.best_score),

        "validation_log_score":
            -float(model.best_score),

        "best_iteration":
            model.best_iteration + 1
    })

    del model
    gc.collect()


elapsed = time.time() - start_time

print(
    f"\nFinished in {elapsed/60:.1f} minutes."
)

joint_results_df = pd.DataFrame(joint_results)

In [ ]:
top_joint = (
    joint_results_df
    .sort_values(
        "validation_log_score",
        ascending=False
    )
    .head(15)
)

top_joint

In [ ]:
best_joint = (
    joint_results_df
    .sort_values(
        "validation_log_score",
        ascending=False
    )
    .iloc[0]
)

best_joint

In [ ]:
# ============================================================
# Final XGBoost check: learning-rate sensitivity
# ============================================================

eta_values = [0.03, 0.05, 0.08, 0.10]

eta_results = []

final_base_params = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",

    "max_depth": 4,
    "min_child_weight": 1.0,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_lambda": 0.25,
    "reg_alpha": 0.0,

    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}


for eta in eta_values:

    print(f"Testing eta = {eta}")

    params = final_base_params.copy()
    params["eta"] = eta

    model = xgb.train(
        params=params,
        dtrain=dtrain_final,

        # Smaller eta may require more rounds
        num_boost_round=4000,

        evals=[(dval_final, "validation")],

        early_stopping_rounds=150,
        verbose_eval=False
    )

    eta_results.append({
        "eta": eta,
        "validation_log_score":
            -float(model.best_score),
        "best_iteration":
            model.best_iteration + 1
    })


eta_results_df = (
    pd.DataFrame(eta_results)
    .sort_values(
        "validation_log_score",
        ascending=False
    )
)

eta_results_df

In [ ]:
# ============================================================
# Training and validation loss curves for final XGBoost model
# ============================================================

import xgboost as xgb
import matplotlib.pyplot as plt
import numpy as np

final_xgb_params = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",

    "max_depth": 4,
    "min_child_weight": 1.0,

    "subsample": 0.85,
    "colsample_bytree": 0.85,

    "reg_lambda": 0.25,
    "reg_alpha": 0.0,

    "eta": 0.08,

    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}

evals_result = {}

final_xgb_curve = xgb.train(
    params=final_xgb_params,
    dtrain=dtrain_final,
    num_boost_round=4000,

    # IMPORTANT:
    # validation is last, so early stopping is based on validation loss
    evals=[
        (dtrain_final, "train"),
        (dval_final, "validation")
    ],

    early_stopping_rounds=150,
    evals_result=evals_result,
    verbose_eval=False
)

print("Best iteration (0-based):", final_xgb_curve.best_iteration)
print("Best number of trees:", final_xgb_curve.best_iteration + 1)
print("Best validation nloglik:", final_xgb_curve.best_score)
print("Best validation log score:", -float(final_xgb_curve.best_score))

In [ ]:
train_loss = evals_result["train"]["poisson-nloglik"]
val_loss = evals_result["validation"]["poisson-nloglik"]

iterations = np.arange(1, len(train_loss) + 1)

best_round = final_xgb_curve.best_iteration + 1

plt.figure(figsize=(8, 5))

plt.plot(
    iterations,
    train_loss,
    label="Training loss"
)

plt.plot(
    iterations,
    val_loss,
    label="Validation loss"
)

plt.axvline(
    best_round,
    linestyle="--",
    label=f"Best iteration = {best_round}"
)

plt.xlabel("Boosting iteration")
plt.ylabel("Poisson negative log-likelihood")
plt.title("XGBoost Training and Validation Loss")
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    iterations,
    train_loss,
    label="Training loss"
)

plt.plot(
    iterations,
    val_loss,
    label="Validation loss"
)

plt.axvline(
    best_round,
    linestyle="--",
    label=f"Best iteration = {best_round}"
)

plt.xlim(1, 180)

plt.xlabel("Boosting iteration")
plt.ylabel("Poisson negative log-likelihood")
plt.title("XGBoost Loss Around the Optimal Boosting Iteration")
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Sanity checks for the selected XGBoost model
# Use ONLY the best 103 boosting iterations
# ============================================================

import numpy as np
import pandas as pd

best_round = final_xgb_curve.best_iteration + 1

pred_train = final_xgb_curve.predict(
    dtrain_final,
    iteration_range=(0, best_round)
)

pred_val = final_xgb_curve.predict(
    dval_final,
    iteration_range=(0, best_round)
)

print("Number of trees used:", best_round)
print("Train predictions:", pred_train.shape)
print("Validation predictions:", pred_val.shape)

In [ ]:
if "backward_path_df" in globals():
    backward_path_df.to_csv(TABLE_DIR / "xgb_backward_elimination_path.csv", index=False)
if "best_complete" in globals():
    best_complete.to_csv(TABLE_DIR / "xgb_feature_set_best_configs.csv", index=False)
if "local_window_results_full" in globals():
    local_window_results_full.to_csv(TABLE_DIR / "xgb_local_window_sensitivity.csv", index=False)
if "joint_results_df" in globals():
    joint_results_df.to_csv(TABLE_DIR / "xgb_joint_local_refinement.csv", index=False)
if "eta_results" in globals():
    pd.DataFrame(eta_results).to_csv(TABLE_DIR / "xgb_learning_rate_sensitivity.csv", index=False)
print("Saved available selection tables:", TABLE_DIR)